<a href="https://colab.research.google.com/github/Justin21523/edge-deid-studio/blob/feature%2Fner-pipeline/notebooks/07_ner_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==================================================
# 第一部分：環境設置與套件安裝
# ==================================================

# 最前面只要執行一次，升級底層套件，解決 glob/fsspec 與 Parquet bug
!pip install -q fsspec==2023.9.2 s3fs gcsfs pyarrow datasets huggingface_hub
!pip install -q transformers accelerate evaluate tensorboard
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.4/173.4 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 8.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-cupti-cu12 12.5.82 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-nvrtc-cu12==12.4.

In [8]:
# 登入Hugging Face（需要存取權杖）
from huggingface_hub import notebook_login
notebook_login()  # 點擊出現的連結，複製權杖貼入

In [9]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

CUDA available: False


In [10]:
from getpass import getpass
import os

# 把 HF 模型與資料快取都放到 Drive，避免重啟就消失
os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_home'
os.environ['HF_DATASETS_CACHE'] = '/content/drive/MyDrive/hf_datasets'
os.environ['HF_HUB_CACHE'] = '/content/drive/MyDrive/hf_hub'

In [4]:
# 1. Clone 兩個 repo 到 /content 底下
!git clone https://github.com/Sripaad/ai4privacy.git /content/ai4privacy
!git clone https://github.com/hemingkx/CLUENER2020.git /content/CLUENER2020

Cloning into '/content/ai4privacy'...
remote: Enumerating objects: 118, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 118 (delta 1), reused 1 (delta 0), pack-reused 113 (from 1)
Receiving objects: 100% (118/118), 77.59 MiB | 9.02 MiB/s, done.
Resolving deltas: 100% (46/46), done.
Updating files: 100% (40/40), done.
Error downloading object: notebooks/ai4p_adapter_en/pytorch_adapter.bin (c45a370): Smudge error: Error downloading notebooks/ai4p_adapter_en/pytorch_adapter.bin (c45a370ed55652ee6bfccc4bb969dfb0325c2e958b9f502d29d7a068a27669b8): batch response: Git LFS is disabled for this repository.

Errors logged to /content/ai4privacy/.git/lfs/logs/20250716T065747.661287548.log
Use `git lfs logs last` to view the log.
error: external filter 'git-lfs filter-process' failed
fatal: notebooks/ai4p_adapter_en/pytorch_adapter.bin: smudge filter lfs failed
You can inspect what was checked out with 'git status'
and retry with 'git 

In [5]:
# 先安裝，如果沒裝過
!pip install -q pandas datasets

import pandas as pd
from datasets import Dataset

# 1) PII-masking-200k JSONL
pii_path = "/content/ai4privacy/data/pii200k_english.jsonl"
# pandas 直接讀 json lines
df_pii = pd.read_json(pii_path, lines=True)
# 轉成 Dataset
pii_ds = Dataset.from_pandas(df_pii)
# 移掉多餘的 __index_level_0__ 欄位（pandas index）

print(pii_ds)
print(pii_ds[0])

Dataset({
    features: ['masked_text', 'unmasked_text', 'token_entity_labels', 'tokenised_unmasked_text'],
    num_rows: 59392
})
{'masked_text': '[PREFIX_1] [FIRSTNAME_1] [LASTNAME_1], please do not forget your appointment on [DATE_1] at [TIME_1]. Keep our [PHONEIMEI_1] and [USERNAME_1] in mind.', 'unmasked_text': 'Ms. Savion Von, please do not forget your appointment on 28th November at 09. Keep our 69-616376-555417-0 and Gilbert_Dooley in mind.', 'token_entity_labels': ['B-PREFIX', 'I-PREFIX', 'B-FIRSTNAME', 'I-FIRSTNAME', 'I-FIRSTNAME', 'B-LASTNAME', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-DATE', 'I-DATE', 'O', 'B-TIME', 'O', 'O', 'O', 'B-PHONEIMEI', 'I-PHONEIMEI', 'I-PHONEIMEI', 'I-PHONEIMEI', 'I-PHONEIMEI', 'I-PHONEIMEI', 'I-PHONEIMEI', 'I-PHONEIMEI', 'I-PHONEIMEI', 'I-PHONEIMEI', 'I-PHONEIMEI', 'I-PHONEIMEI', 'O', 'B-USERNAME', 'I-USERNAME', 'I-USERNAME', 'I-USERNAME', 'O', 'O', 'O'], 'tokenised_unmasked_text': ['ms', '.', 'sa', '##vio', '##n', 'von', ',', 'please', 'do', 'n

In [6]:
import os
import json
import pandas as pd
from datasets import Dataset, DatasetDict

BASE = "/content/CLUENER2020/BERT-Softmax/data/clue"
paths = {
    "train": os.path.join(BASE, "train.json"),
    "test": os.path.join(BASE, "test.json"),
}

def clean_json_line(line):
    # 處理常見的 JSON 格式問題
    line = line.replace("'", '"')  # 單引號轉雙引號
    line = line.replace("None", "null")  # Python None 轉 JSON null
    return line


def load_jsonl_to_df(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = clean_json_line(line.strip())
            if not line:
                continue
            try:
                data = json.loads(line)
                # 確保每個記錄都有text和label字段
                if "text" in data and "label" in data:
                    records.append({
                        "text": data["text"],
                        "label": data["label"]
                    })
            except json.JSONDecodeError:
                continue
    return pd.DataFrame(records)

# 加載數據到DataFrame
dfs = {split: load_jsonl_to_df(p) for split, p in paths.items()}

# 轉換標籤函數 - 修復NoneType錯誤的版本
def convert_labels(example):
    ents = []
    try:
        if "label" in example and example["label"]:
            for ent_type, ent_dict in example["label"].items():
                if ent_dict is None or not ent_dict:
                    continue

                for entity_name, positions in ent_dict.items():
                    if positions is None:
                        continue

                    if not isinstance(positions, list):
                        continue

                    for pos in positions:
                        try:
                            if isinstance(pos, list) and len(pos) == 2:
                                start, end = pos
                                if isinstance(start, int) and isinstance(end, int):
                                    ents.append({
                                        "entity": ent_type,
                                        "start": start,
                                        "end": end
                                    })
                        except:
                            # 跳過單個位置錯誤
                            continue
    except Exception as e:
        # 捕獲並記錄錯誤，但繼續處理其他數據
        print(f"處理實體時出錯: {e}")
        print(f"問題數據: {example}")

    return {"entities": ents}
# 創建Dataset
datasets = {}
for split, df in dfs.items():
    ds = Dataset.from_pandas(df)

    # 移除自動添加的索引列
    if "__index_level_0__" in ds.column_names:
        ds = ds.remove_columns("__index_level_0__")

    # 應用標籤轉換
    ds = ds.map(
        convert_labels,
        remove_columns=["label"],  # 移除原始標籤列
        batched=False,
        load_from_cache_file=False  # 避免緩存問題
    )
    datasets[split] = ds

# 創建DatasetDict
cluener_ds = DatasetDict(datasets)

# 檢查結果
print(cluener_ds)
print("\nTrain 首行：")
print(cluener_ds["train"][0])

# 驗證實體轉換
sample = cluener_ds["train"][0]
print("\n文本內容:", sample["text"])
print("實體列表:", sample["entities"])

Map:   0%|          | 0/10738 [00:00<?, ? examples/s]

Map:   0%|          | 0/1340 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'entities'],
        num_rows: 10738
    })
    test: Dataset({
        features: ['text', 'entities'],
        num_rows: 1340
    })
})

Train 首行：
{'text': '浙商银行企业信贷部叶老桂博士则从另一个角度对五道门槛进行了解读。叶老桂认为，对目前国内商业银行而言，', 'entities': [{'end': 3, 'entity': 'company', 'start': 0}, {'end': 11, 'entity': 'name', 'start': 9}, {'end': 34, 'entity': 'name', 'start': 32}]}

文本內容: 浙商银行企业信贷部叶老桂博士则从另一个角度对五道门槛进行了解读。叶老桂认为，对目前国内商业银行而言，
實體列表: [{'end': 3, 'entity': 'company', 'start': 0}, {'end': 11, 'entity': 'name', 'start': 9}, {'end': 34, 'entity': 'name', 'start': 32}]


In [12]:
cluener_ds["train"]

Dataset({
    features: ['text', 'entities'],
    num_rows: 10738
})

In [13]:
from transformers import AutoTokenizer
import numpy as np

# 載入 tokenizer
tokenizer = AutoTokenizer.from_pretrained("ckiplab/bert-base-chinese-ner", use_fast=True)

# 步驟1: 收集所有可能的實體類型
# 從訓練集中提取所有出現過的實體類型
all_entity_types = set()

for example in cluener_ds["train"]:
    for entity in example["entities"]:
        all_entity_types.add(entity["entity"])

# 轉換為排序後的列表
entity_types = sorted(list(all_entity_types))

# 步驟2: 創建標籤映射（使用BIO標註方案）
labels = ["O"]  # "O" 表示非實體
for ent_type in entity_types:
    labels.append(f"B-{ent_type}")  # 實體開始
    labels.append(f"I-{ent_type}")  # 實體內部

label2id = {label: idx for idx, label in enumerate(labels)}
id2label = {idx: label for idx, label in enumerate(labels)}

print("標籤列表:", labels)

# 步驟3: 修改標籤對齊函數
def tokenize_and_align_labels(examples):
    # 1. tokenize 文本
    tokenized_inputs = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128,
        return_offsets_mapping=True  # 需要這個來對齊標籤
    )

    # 2. 獲取偏移映射
    offset_mapping = tokenized_inputs.pop("offset_mapping")

    # 3. 初始化標籤列表
    labels_list = []

    # 4. 為每個樣本處理
    for i, (text, entities, offsets) in enumerate(zip(
        examples["text"],
        examples["entities"],
        offset_mapping
    )):
        # 創建一個與文本長度相同的字符級標籤陣列，初始化為"O"（非實體）
        char_labels = ["O"] * len(text)

        # 標記所有實體位置
        for entity in entities:
            start = entity["start"]
            end = entity["end"]
            ent_type = entity["entity"]

            # 標記實體開始位置
            if start < len(char_labels):
                char_labels[start] = f"B-{ent_type}"

            # 標記實體內部位置
            for pos in range(start + 1, end):
                if pos < len(char_labels):
                    char_labels[pos] = f"I-{ent_type}"

        # 將字符級標籤轉換為token級標籤
        token_labels = []
        for (start, end) in offsets:
            if start == end == 0:  # 特殊token (如[CLS], [SEP], padding)
                token_labels.append(-100)  # 在損失計算中忽略
            else:
                # 取token覆蓋的第一個字符的標籤
                token_labels.append(label2id.get(char_labels[start], label2id["O"]))

        labels_list.append(token_labels)

    tokenized_inputs["labels"] = labels_list
    return tokenized_inputs

# 步驟4: 應用處理
train_tok = cluener_ds["train"].map(tokenize_and_align_labels, batched=True)
test_tok = cluener_ds["test"].map(tokenize_and_align_labels, batched=True)

# 步驟5: 檢查結果
print("\n處理後的訓練集樣本:")
print("文本:", train_tok[0]["text"])
print("Token IDs:", train_tok[0]["input_ids"][:20])
print("標籤:", train_tok[0]["labels"][:20])
print("對應標籤文字:", [id2label.get(idx, "IGNORE") for idx in train_tok[0]["labels"][:20]])

# 步驟6: 驗證對齊
def visualize_alignment(example_idx=0):
    tokens = tokenizer.convert_ids_to_tokens(train_tok[example_idx]["input_ids"])
    labels = [id2label.get(idx, "IGNORE") for idx in train_tok[example_idx]["labels"]]

    print("\n文本:", train_tok[example_idx]["text"])
    print("Token與標籤對齊:")
    for token, label in zip(tokens, labels):
        print(f"{token}\t{label}")

visualize_alignment()

標籤列表: ['O', 'B-address', 'I-address', 'B-book', 'I-book', 'B-company', 'I-company', 'B-game', 'I-game', 'B-government', 'I-government', 'B-movie', 'I-movie', 'B-name', 'I-name', 'B-organization', 'I-organization', 'B-position', 'I-position', 'B-scene', 'I-scene']


Map:   0%|          | 0/10738 [00:00<?, ? examples/s]

Map:   0%|          | 0/1340 [00:00<?, ? examples/s]


處理後的訓練集樣本:
文本: 浙商银行企业信贷部叶老桂博士则从另一个角度对五道门槛进行了解读。叶老桂认为，对目前国内商业银行而言，
Token IDs: [101, 3851, 1555, 7213, 6121, 821, 689, 928, 6587, 6956, 1383, 5439, 3424, 1300, 1894, 1156, 794, 1369, 671, 702]
標籤: [-100, 5, 6, 6, 0, 0, 0, 0, 0, 0, 13, 14, 0, 0, 0, 0, 0, 0, 0, 0]
對應標籤文字: ['IGNORE', 'B-company', 'I-company', 'I-company', 'O', 'O', 'O', 'O', 'O', 'O', 'B-name', 'I-name', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']

文本: 浙商银行企业信贷部叶老桂博士则从另一个角度对五道门槛进行了解读。叶老桂认为，对目前国内商业银行而言，
Token與標籤對齊:
[CLS]	IGNORE
浙	B-company
商	I-company
银	I-company
行	O
企	O
业	O
信	O
贷	O
部	O
叶	B-name
老	I-name
桂	O
博	O
士	O
则	O
从	O
另	O
一	O
个	O
角	O
度	O
对	O
五	O
道	O
门	O
槛	O
进	O
行	O
了	O
解	O
读	O
。	O
叶	B-name
老	I-name
桂	O
认	O
为	O
，	O
对	O
目	O
前	O
国	O
内	O
商	O
业	O
银	O
行	O
而	O
言	O
，	O
[SEP]	IGNORE
[PAD]	IGNORE
[PAD]	IGNORE
[PAD]	IGNORE
[PAD]	IGNORE
[PAD]	IGNORE
[PAD]	IGNORE
[PAD]	IGNORE
[PAD]	IGNORE
[PAD]	IGNORE
[PAD]	IGNORE
[PAD]	IGNORE
[PAD]	IGNORE
[PAD]	IGNORE
[PAD]	IGNORE
[PAD]	IGNORE
[PAD]	IGNORE
[PAD]	IGNORE
[PAD]	IGNORE
[PAD]	IGNORE
[PAD]

In [ ]:
from transformers import AutoModelForTokenClassification, DataCollatorForTokenClassification, TrainingArguments, Trainer
import evaluate, numpy as np

# 載入模型
model = AutoModelForTokenClassification.from_pretrained(
    "ckiplab/bert-base-chinese-ner",
    num_labels=len(labels),
    id2label={i:l for l,i in label2id.items()},
    label2id=label2id,
)

# Data collator 與指標
data_collator = DataCollatorForTokenClassification(tokenizer)
metric = evaluate.load("seqeval")
def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=2)
    true = p.label_ids
    # strip -100
    true_labels = [[labels[t] for t in row if t!=-100] for row in true]
    pred_labels = [[labels[p] for (p,t) in zip(row_p,row_t) if t!=-100]
                   for row_p,row_t in zip(preds,true)]
    res = metric.compute(predictions=pred_labels, references=true_labels)
    return {"precision": res["overall_precision"], "recall": res["overall_recall"], "f1": res["overall_f1"]}

# TrainingArguments
args = TrainingArguments(
    output_dir="drive/MyDrive/edge-deid/models/ner/v1.0",
    evaluation_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    learning_rate=5e-5,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=train_tok,      # demo only；實務請用 validation split
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()
trainer.save_model()


In [ ]:
import torch, onnxruntime
from onnxruntime.quantization import quantize_dynamic, QuantType

# 1) 先載 PyTorch 模型
model = AutoModelForTokenClassification.from_pretrained("drive/MyDrive/edge-deid/models/ner/v1.0")
model.eval()

# 2) 建 dummy input
dummy = tokenizer("測試用文字", return_tensors="pt", padding="max_length", max_length=128)

# 3) 匯出 ONNX
torch.onnx.export(
    model,
    (dummy["input_ids"], dummy["attention_mask"]),
    "drive/MyDrive/edge-deid/onnx/ner.onnx",
    input_names=["input_ids","attention_mask"],
    output_names=["logits"],
    dynamic_axes={"input_ids":{0:"batch"}, "attention_mask":{0:"batch"}, "logits":{0:"batch"}},
    opset_version=13,
)

# 4) 動態量化
quantize_dynamic(
    "drive/MyDrive/edge-deid/onnx/ner.onnx",
    "drive/MyDrive/edge-deid/onnx/ner_int8.onnx",
    weight_type=QuantType.QInt8,
)

In [ ]:
# Colab 第六格：對測試集做評估並視覺化 F1 結果
# 重新載入 ONNX 模型測試推論（示範單例）
ort_sess = onnxruntime.InferenceSession("drive/MyDrive/edge-deid/onnx/ner_int8.onnx")
inputs  = {k: v.cpu().numpy() for k,v in dummy_input.items()}
logits  = ort_sess.run(None, inputs)[0]
pred_ids = np.argmax(logits, axis=-1)[0]
print([id2label[i] for i in pred_ids])

# 用原 Trainer.predict 做整體指標
pred_results = trainer.predict(train_ds)
print(pred_results.metrics)

# （選做）用 matplotlib 畫出不同 epoch F1 變化曲線
import matplotlib.pyplot as plt
f1s = [m["eval_f1"] for m in trainer.state.log_history if "eval_f1" in m]
plt.plot(f1s)
plt.title("Epoch vs F1")
plt.xlabel("Epoch")
plt.ylabel("F1")
plt.show()
